In [34]:
import pandas as pd
import configparser
import time
import random
import json
from sentence_transformers import SentenceTransformer
from pymilvus import connections, Collection, FieldSchema, CollectionSchema, DataType, utility

In [35]:
# df_ds = pd.read_json('data-ds.json', encoding = 'utf-8')
df_ml = pd.read_json('data-ml.json', orient='record')
df_ai = pd.read_json('data-ai.json', orient='record')
df = pd.concat([df_ai, df_ml], axis=0)

df_select = df[
      ['job_id','job_title', 'employer_name', 'employer_logo', 'employer_website',
       'employer_company_type', 'job_publisher', 'job_employment_type',
       'job_apply_link', 'job_description',
       'job_is_remote', 'job_city', 'job_state',
       'job_latitude', 'job_longitude', 'job_benefits',
       'job_required_experience', 'job_required_skills',
       'job_required_education', 'job_experience_in_place_of_education',
       'job_highlights']
].copy()


def clean_job_postings(df):
  df['job_location'] = df['job_city'] + ', ' + df['job_state']
  df['info'] = df['job_title'] + '|' + df['job_location'] + '|' + df['employer_name']
  df = df.drop_duplicates(subset='info', ignore_index=True)
  df = df.drop_duplicates(subset='job_description', ignore_index=True)
  df = df[~df['job_publisher'].str.contains('Geebo')] # filter out job postings from geebo.com
  df_exp = pd.json_normalize(df['job_required_experience'])
  # df = pd.concat([df.drop(columns=['job_required_experience']), df_exp], axis=1)
  df = pd.concat([df, df_exp], axis=1)
  df['required_experience_in_months'] = df['required_experience_in_months'].fillna(0.0)
  df['required_experience'] = df['required_experience_in_months'].astype(int) / 12
  df['citizenship'] = df['job_description'].str.contains('clearance') | (df['job_description'].str.contains('SCI')) | (df['job_description'].str.contains('US citizenship'))| (df['job_description'].str.contains('Clearance')) | (df['job_description'].str.contains('US Citizen'))
  return df

df_select = clean_job_postings(df_select)
# Set up the DataFrame
job_postings = df_select
job_postings = job_postings.dropna(subset=['info', 'job_description'])
job_postings = job_postings.fillna('')

In [21]:
job_postings.head(2)

,job_id,job_title,employer_name,employer_logo,employer_website,employer_company_type,job_publisher,job_employment_type,job_apply_link,job_description,...,job_experience_in_place_of_education,job_highlights,job_location,info,no_experience_required,required_experience_in_months,experience_mentioned,experience_preferred,required_experience,citizenship
0,RdI79DIBlTXe-BmsAAAAAA==,AI Math System Trainer,Outlier,https://encrypted-tbn0.gstatic.com/images?q=tb...,,,LinkedIn,PARTTIME,https://www.linkedin.com/jobs/view/ai-math-sys...,Outlier helps the worlds most innovative compa...,...,False,{'Qualifications': ['A bachelor's or higher de...,"Air Force Academy, CO","AI Math System Trainer|Air Force Academy, CO|O...",false,0.0,true,false,0.0,False
2,5Nsyzr4-o2nu4dZQAAAAAA==,Patent Agent with Expertise in AI and Machine ...,Premier Legal Staffing,,http://premierlegalstaffing.com,,Get.It,FULLTIME,https://www.get.it/job/patent-agent-with-exper...,Description\n\nA leading general practice firm...,...,False,{'Qualifications': ['The ideal candidate shoul...,"Englewood, CO",Patent Agent with Expertise in AI and Machine ...,false,0.0,true,false,0.0,False


In [37]:
if __name__ == '__main__':
    # connect to milvus
    cfp = configparser.RawConfigParser()
    cfp.read('config.ini')
    milvus_uri = cfp.get('jobmatch', 'uri')
    token = cfp.get('jobmatch', 'token')
    connections.connect("default",
                        uri=milvus_uri,
                        token=token)
    print(f"Connecting to DB: {milvus_uri}")


    # Initialize embedding model
    embedder = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Define Milvus collection schema
    fields = [
        FieldSchema(name="job_id", dtype=DataType.VARCHAR, is_primary=True, max_length=100),
        FieldSchema(name="job_description_vector", dtype=DataType.FLOAT_VECTOR, dim=384),
        FieldSchema(name="job_description_raw", dtype=DataType.VARCHAR, max_length=50000),
        FieldSchema(name="info", dtype=DataType.VARCHAR, max_length=255),
        FieldSchema(name="required_experience", dtype=DataType.FLOAT, max_length=100),
        FieldSchema(name="citizenship", dtype=DataType.BOOL, max_length=100),
        FieldSchema(name="job_apply_link", dtype=DataType.VARCHAR, max_length=255)
    ]
    
    schema = CollectionSchema(fields, description="Job listing data with job description embeddings and metadata")
    
    # Create collection (if it doesn't already exist)
    collection_name = "job_listings"
    if collection_name not in utility.list_collections():
        job_collection = Collection(name=collection_name, schema=schema)
    else:
        job_collection = Collection(name=collection_name)
    

    
    # Prepare lists for each field in the schema to match Milvus requirements
    id_list = []
    vector_list = []
    description_list = []
    info_list = []
    experience_list = []
    citizenship_list = []
    link_list = []
    
    # Populate lists with DataFrame data
    for index, row in job_postings.iterrows():
        job_description_vector = embedder.encode(row['job_description']).tolist()
        
        # Append data for each field
        id_list.append(row['job_id'])  # Optional if using auto-generated ID in Milvus
        vector_list.append(job_description_vector)
        description_list.append(row['job_description'])
        info_list.append(row['info'])
        experience_list.append(row['required_experience'])
        citizenship_list.append(row['citizenship'])
        link_list.append(row['job_apply_link'])
    
    # Insert data into Milvus collection
    job_collection.insert([
        id_list,
        vector_list,
        description_list,
        info_list,
        experience_list,
        citizenship_list,
        link_list
    ])
    
    print("DataFrame indexed into Milvus successfully.")

Connecting to DB: https://in03-3acf9d8039d538c.serverless.gcp-us-west1.cloud.zilliz.com
DataFrame indexed into Milvus successfully.
